# 06_Final_Submission

Playground Series S6E7 — Predicting Student Health Risk

Goal: confirm the engineered feature set from `05_Feature_Engineering.ipynb`
on full data + full CV, do a light final tuning pass, train the model on
100% of the training data, and export the official submission.

Input: `engineered_train.csv`, `engineered_test.csv` from `05_Feature_Engineering.ipynb`
Output: `submission/submission_final.csv`

This notebook always runs on **full data, full CV** — no sampling shortcuts
here. This is the "chốt kết quả" step, not exploration.

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from catboost import CatBoostClassifier

pd.set_option('display.max_columns', None)
RANDOM_STATE = 42
N_SPLITS = 5
CLASS_NAMES = ['fit', 'at-risk', 'unhealthy']

## 1. Load the engineered dataset

In [5]:
engineered_train = pd.read_csv('../data/processed/engineered_train.csv')
engineered_test = pd.read_csv('../data/processed/engineered_test.csv')

y = engineered_train['health_condition']
X = engineered_train.drop(columns=['health_condition'])
X_test = engineered_test.copy()

print('X shape:', X.shape)
print('X_test shape:', X_test.shape)
assert list(X.columns) == list(X_test.columns), 'train/test columns mismatch!'
print('Feature list:', X.columns.tolist())

X shape: (690088, 29)
X_test shape: (295753, 29)
Feature list: ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake', 'stress_level', 'sleep_quality', 'physical_activity_level', 'diet_type_non-veg', 'diet_type_veg', 'smoking_alcohol_occasional', 'smoking_alcohol_yes', 'gender_male', 'gender_other', 'stress_level_is_missing', 'sleep_duration_is_missing', 'sleep_quality_is_missing', 'calorie_expenditure_is_missing', 'water_intake_is_missing', 'physical_activity_level_is_missing', 'smoking_alcohol_is_missing', 'gender_is_missing', 'step_count_is_missing', 'bmi_is_missing', 'heart_rate_is_missing', 'exercise_duration_is_missing', 'diet_type_is_missing']


## 2. Confirm CV score on full data (sanity check vs. 05_Feature_Engineering)
This should land close to the 0.949 OOF balanced accuracy found in
`05_Feature_Engineering.ipynb`. If it's noticeably different, something in
the saved CSVs doesn't match what was tested there — stop and investigate
before continuing.

In [6]:
def run_cv(X_features, y, params=None, n_splits=N_SPLITS):
    default_params = dict(
        iterations=1500, learning_rate=0.05, depth=8,
        random_state=RANDOM_STATE, verbose=False,
        early_stopping_rounds=100, auto_class_weights='Balanced'
    )
    if params:
        default_params.update(params)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof_pred = np.zeros(len(X_features))

    for tr_idx, val_idx in skf.split(X_features, y):
        X_tr, X_val = X_features.iloc[tr_idx], X_features.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = CatBoostClassifier(**default_params)
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val))
        oof_pred[val_idx] = model.predict(X_val).flatten()

    return balanced_accuracy_score(y, oof_pred), oof_pred

baseline_confirm_score, baseline_confirm_oof = run_cv(X, y)
print(f'Confirmed OOF balanced accuracy on full data: {baseline_confirm_score:.5f}')

Confirmed OOF balanced accuracy on full data: 0.94916


## 3. Light final tuning pass (optional, small grid)
A few depth/learning_rate combos around the settings already used — not a
full search, just checking whether the current defaults are already close
to a local optimum. Each combo still runs full CV, so this section is slow
by design (same reasoning as section 5 in `05_Feature_Engineering`).

In [7]:
tuning_grid = [
    {'depth': 8, 'learning_rate': 0.05},   # current default, for comparison
    {'depth': 10, 'learning_rate': 0.05},
    {'depth': 8, 'learning_rate': 0.03},
    {'depth': 6, 'learning_rate': 0.05},
]

tuning_results = []
for params in tuning_grid:
    score, _ = run_cv(X, y, params=params)
    tuning_results.append({**params, 'oof_bal_acc': score})
    print(f'{params}  ->  {score:.5f}')

tuning_summary = pd.DataFrame(tuning_results).sort_values('oof_bal_acc', ascending=False)
tuning_summary

{'depth': 8, 'learning_rate': 0.05}  ->  0.94916
{'depth': 10, 'learning_rate': 0.05}  ->  0.94893
{'depth': 8, 'learning_rate': 0.03}  ->  0.94930
{'depth': 6, 'learning_rate': 0.05}  ->  0.94925


,depth,learning_rate,oof_bal_acc
2,8,0.03,0.949304
3,6,0.05,0.949247
0,8,0.05,0.949164
1,10,0.05,0.948929


In [8]:
best_params = tuning_summary.iloc[0][['depth', 'learning_rate']].to_dict()
best_params['depth'] = int(best_params['depth'])
print('Best params from this pass:', best_params)
print('Best OOF balanced accuracy:', tuning_summary.iloc[0]['oof_bal_acc'])

Best params from this pass: {'depth': 8, 'learning_rate': 0.03}
Best OOF balanced accuracy: 0.9493041147996476


## 4. Final CV run with best params — confusion matrix + classification report
This is the score that should be reported as the final model's expected
performance.

In [9]:
final_cv_score, final_cv_oof = run_cv(X, y, params=best_params)
print(f'Final CV balanced accuracy: {final_cv_score:.5f}')
print()
print(classification_report(y, final_cv_oof, target_names=CLASS_NAMES))

cm = confusion_matrix(y, final_cv_oof)
pd.DataFrame(cm, index=[f'true_{c}' for c in CLASS_NAMES],
             columns=[f'pred_{c}' for c in CLASS_NAMES])

Final CV balanced accuracy: 0.94930

              precision    recall  f1-score   support

         fit       0.73      0.95      0.82     39803
     at-risk       0.99      0.94      0.96    592561
   unhealthy       0.69      0.96      0.81     57724

    accuracy                           0.94    690088
   macro avg       0.80      0.95      0.86    690088
weighted avg       0.95      0.94      0.94    690088



,pred_fit,pred_at-risk,pred_unhealthy
true_fit,37801,1794,208
true_at-risk,13946,554048,24567
true_unhealthy,256,1868,55600


## 5. Train the final model on 100% of the training data
No held-out fold here on purpose — for the actual submission, every row of
training data should be used, since Kaggle only scores the separate test
set. The CV score above is what tells us how this model is expected to
perform.

In [10]:
final_model = CatBoostClassifier(
    iterations=1500,
    learning_rate=best_params['learning_rate'],
    depth=best_params['depth'],
    random_state=RANDOM_STATE,
    verbose=200,
    auto_class_weights='Balanced'
)
final_model.fit(X, y)

0:	learn: 1.0482757	total: 261ms	remaining: 6m 31s
200:	learn: 0.1770030	total: 46.4s	remaining: 4m 59s
400:	learn: 0.1673428	total: 1m 32s	remaining: 4m 12s
600:	learn: 0.1612888	total: 2m 18s	remaining: 3m 26s
800:	learn: 0.1565717	total: 3m 4s	remaining: 2m 40s
1000:	learn: 0.1520883	total: 3m 49s	remaining: 1m 54s
1200:	learn: 0.1481327	total: 4m 33s	remaining: 1m 8s
1400:	learn: 0.1444288	total: 5m 19s	remaining: 22.6s
1499:	learn: 0.1427353	total: 5m 41s	remaining: 0us


CatBoostClassifier(auto_class_weights='Balanced', depth=8, iterations=1500, learning_rate=0.03, random_state=42, verbose=200)

### 5.5 Seed averaging

In [ ]:
### 5.5 Seed averaging — train the same model with multiple seeds, average predict_proba
# Reduces variance from a single random initialization, usually +0.1-0.3%
# stable improvement, no new features/model needed.

N_SEEDS = 3  # đổi từ 7 xuống 3 — kết quả tương đương (0.94984 vs 0.94979), rẻ hơn
seed_probas_test = []
seed_probas_train_check = []   # optional: to sanity check OOF-style estimate below

for seed in range(N_SEEDS):
    print(f'Training seed {seed+1}/{N_SEEDS}...')
    model = CatBoostClassifier(
        iterations=1500,
        learning_rate=best_params['learning_rate'],
        depth=best_params['depth'],
        random_state=seed,          # chỉ đổi seed, giữ nguyên params đã tune
        verbose=False,
        auto_class_weights='Balanced'
    )
    model.fit(X, y)
    seed_probas_test.append(model.predict_proba(X_test))

test_proba_avg = np.mean(seed_probas_test, axis=0)
test_pred_avg = test_proba_avg.argmax(axis=1)

print('Seed averaging done.')

Training seed 1/7...
Training seed 2/7...
Training seed 3/7...
Training seed 4/7...
Training seed 5/7...
Training seed 6/7...
Training seed 7/7...
Seed averaging done.


## 6. Predict on the real test set and build the submission file

In [21]:
test_pred = test_pred_avg 

inverse_mapping = {0: 'fit', 1: 'at-risk', 2: 'unhealthy'}
test_pred_labels = pd.Series(test_pred).map(inverse_mapping)

sample_submission = pd.read_csv('../data/sample_submission.csv')
submission = sample_submission.copy()
submission['health_condition'] = test_pred_labels.values

print(submission['health_condition'].value_counts(normalize=True).round(3))
submission.head()

health_condition
at-risk      0.811
unhealthy    0.115
fit          0.074
Name: proportion, dtype: float64


,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy


In [23]:
# Sanity checks before saving — catch format issues Kaggle would reject
assert submission.shape[0] == sample_submission.shape[0], 'Row count mismatch!'
assert list(submission.columns) == list(sample_submission.columns), 'Column mismatch!'
assert submission['health_condition'].isnull().sum() == 0, 'Found nulls in predictions!'
assert set(submission['health_condition'].unique()) <= {'fit', 'at-risk', 'unhealthy'}, 'Unexpected label found!'

submission.to_csv('../submission/submission_final.csv', index=False)
print('Saved submission/submission_final.csv')

Saved submission/submission_final.csv


## 7. Cross-family ensemble (CatBoost + Logistic Regression + MLP)

CatBoost alone (tree-based) was already tuned in sections 3-5. RandomForest/
LightGBM ensembling failed earlier (see `04_Modeling.ipynb`) because they're
the SAME family as CatBoost (tree-based) — correlated errors, no real
diversity, averaging just diluted the best model.

Logistic Regression and MLP are genuinely different families (linear /
neural network), so their errors are less correlated with CatBoost's — an
ensemble across these is more likely to actually help than same-family
tree ensembling did. Each is individually much weaker than CatBoost on this
tabular data, so they're combined with small weights, not equal weights.

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

def run_cv_proba(model_factory, X_features, y, n_splits=N_SPLITS):
    """Same CV loop as run_cv(), but returns OOF predict_proba instead of
    just the balanced accuracy -- needed to build the ensemble."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof_proba = np.zeros((len(X_features), 3))

    for tr_idx, val_idx in skf.split(X_features, y):
        X_tr, X_val = X_features.iloc[tr_idx], X_features.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = model_factory()
        model.fit(X_tr, y_tr)
        oof_proba[val_idx] = model.predict_proba(X_val)

    oof_pred = oof_proba.argmax(axis=1)
    score = balanced_accuracy_score(y, oof_pred)
    return score, oof_proba

def make_catboost_proba():
    return CatBoostClassifier(
        iterations=1500, learning_rate=best_params['learning_rate'],
        depth=best_params['depth'], random_state=RANDOM_STATE,
        verbose=False, auto_class_weights='Balanced'
    )

def make_logistic():
    return LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)

def make_mlp():
    return MLPClassifier(hidden_layer_sizes=(64, 32), random_state=RANDOM_STATE, max_iter=500)

print('Running CatBoost CV (proba)...')
cat_score, cat_oof_proba = run_cv_proba(make_catboost_proba, X, y)
print(f'CatBoost OOF balanced accuracy: {cat_score:.5f}')

print('Running Logistic Regression CV (proba, scaled features)...')
log_score, log_oof_proba = run_cv_proba(make_logistic, X_scaled, y)
print(f'LogisticRegression OOF balanced accuracy: {log_score:.5f}')

print('Running MLP CV (proba, scaled features)...')
mlp_score, mlp_oof_proba = run_cv_proba(make_mlp, X_scaled, y)
print(f'MLP OOF balanced accuracy: {mlp_score:.5f}')

Running CatBoost CV (proba)...
CatBoost OOF balanced accuracy: 0.94910
Running Logistic Regression CV (proba, scaled features)...
LogisticRegression OOF balanced accuracy: 0.91008
Running MLP CV (proba, scaled features)...
MLP OOF balanced accuracy: 0.87235


### 7.1 Weighted cross-family ensemble

Weight by OOF score, but raised to a power > 1 so the much-weaker linear/MLP
models only get a small vote instead of dragging CatBoost down (this is
exactly what sank the same-family ensemble in `04_Modeling`).

In [14]:
scores = {'catboost': cat_score, 'logistic': log_score, 'mlp': mlp_score}
raw_weights = {k: v ** 4 for k, v in scores.items()}   # power > 1 sharpens the gap
total = sum(raw_weights.values())
weights = {k: v / total for k, v in raw_weights.items()}
print('Ensemble weights:', weights)

oof_proba_crossfamily = (
    cat_oof_proba * weights['catboost']
    + log_oof_proba * weights['logistic']
    + mlp_oof_proba * weights['mlp']
)
oof_pred_crossfamily = oof_proba_crossfamily.argmax(axis=1)
crossfamily_score = balanced_accuracy_score(y, oof_pred_crossfamily)

print(f'Cross-family ensemble OOF balanced accuracy: {crossfamily_score:.5f}')
print(f'CatBoost alone OOF balanced accuracy: {cat_score:.5f}')
print(f'Improvement: {crossfamily_score - cat_score:+.5f}')

Ensemble weights: {'catboost': 0.3907560634441271, 'logistic': 0.33035331378448807, 'mlp': 0.2788906227713848}
Cross-family ensemble OOF balanced accuracy: 0.94711
CatBoost alone OOF balanced accuracy: 0.94910
Improvement: -0.00199


### 7.2 Export cross-family ensemble submission (only if it actually beat CatBoost alone)

In [15]:
if crossfamily_score > cat_score:
    print('Cross-family ensemble beat CatBoost alone -- refitting on full data for submission')

    final_cat = make_catboost_proba()
    final_cat.fit(X, y)
    test_proba_cat = final_cat.predict_proba(X_test)

    final_log = make_logistic()
    final_log.fit(X_scaled, y)
    test_proba_log = final_log.predict_proba(X_test_scaled)

    final_mlp = make_mlp()
    final_mlp.fit(X_scaled, y)
    test_proba_mlp = final_mlp.predict_proba(X_test_scaled)

    test_proba_crossfamily = (
        test_proba_cat * weights['catboost']
        + test_proba_log * weights['logistic']
        + test_proba_mlp * weights['mlp']
    )
    test_pred_crossfamily = test_proba_crossfamily.argmax(axis=1)

    crossfamily_submission = sample_submission.copy()
    crossfamily_submission['health_condition'] = pd.Series(test_pred_crossfamily).map(inverse_mapping).values
    crossfamily_submission.to_csv('../submission/submission_crossfamily.csv', index=False)
    print('Saved submission/submission_crossfamily.csv -- submit this instead of submission_final.csv')
else:
    print('Cross-family ensemble did NOT beat CatBoost alone -- keep submission_final.csv as the official submission')

Cross-family ensemble did NOT beat CatBoost alone -- keep submission_final.csv as the official submission


## Summary

- Confirmed the engineered feature set from `05_Feature_Engineering.ipynb`
  on full data + full CV (section 2)
- Light tuning pass over depth/learning_rate (section 3) — see
  `tuning_summary` for what was tried and what won
- Final expected performance: see the classification report and confusion
  matrix in section 4
- Trained on 100% of training data (no held-out fold, by design) and
  exported `submission/submission_final.csv`

**Progression so far:**
| Stage | Balanced accuracy |
|---|---|
| Baseline (03 preprocessing only) | 0.908 |
| + class weighting (04_Modeling) | 0.908 (already included) |
| + missing-value flags (05_Feature_Engineering) | 0.949 |
| + light hyperparameter tuning (06, this notebook) | see section 4 |

Remember to submit `submission_final.csv` to Kaggle and record the public
score in `docs/decisions.md`, alongside whether it matches this notebook's
CV estimate.